# USA external-validation preprocessing — four GA × PMA-cut outputs

This notebook preserves the original `usa_data` adverse-cohort behavior: **LOS, abdominal NEC, sepsis, and other adverse patients remain in every output**, and their labels are stored only as metadata.

It writes four model-ready PKLs:

1. `ALL_GA_PMA50_PATIENT_CUT`: all GA; remove an entire patient if any source PMA is outside 0–50 weeks.
2. `ALL_GA_PMA50_WINDOW_CUT`: all GA; retain the patient and remove only windows whose target PMA is outside 0–50 weeks.
3. `GA_LE35_PMA50_PATIENT_CUT`: GA≤35; patient-level PMA cut.
4. `GA_LE35_PMA50_WINDOW_CUT`: GA≤35; window-level PMA cut.

The expensive patient conversion is performed once. GA and PMA rules are applied only while consolidating the four final files. Existing old caches are invalidated through the new pipeline version.

## QC rule

For each 120-minute superwindow, QC is calculated **separately for every dynamic feature** across its 23 frames. If **any** dynamic feature has more than 20% missing frames, the entire superwindow is rejected. With 23 frames, this means each dynamic feature may have at most 4 missing frames; 5 or more missing frames causes rejection. Remaining missing values in retained windows are handled by the existing per-feature linear interpolation and mask-channel logic. Static GA, sex, and birthweight are excluded from this QC.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Build the USA external-validation dataset for the frozen Sweden-trained
common-feature PMA models.

Input
-----
One parquet per patient under:
    /mnt/home/qinqiu/thesis/data/US_patient_features/patients
with an optional matching cache JSON:
    <pid>.parquet.cache.json

Output
------
Four model-ready pickles forming the full cross-product of:
- GA unrestricted versus GA <=35
- PMA outliers handled at patient level versus window level

The four cohorts are:
1. ALL_GA + PMA50_PATIENT_CUT
2. ALL_GA + PMA50_WINDOW_CUT
3. GA_LE35 + PMA50_PATIENT_CUT
4. GA_LE35 + PMA50_WINDOW_CUT

Each output contains:
    X_by_patient[pid]  : (N, 23, 17) float32, Sweden-compatible raw windows
    X2_by_patient[pid] : (N, 2, 23, 17) float32
    y_by_patient[pid]  : (N,) true PMA in weeks
    meta_by_patient[pid] : basic window metadata DataFrame
    patient_metadata_by_patient[pid] : patient/adverse metadata dict
    window_adverse_metadata_by_patient[pid] : adverse/time-to-event metadata

The preprocessing deliberately reproduces the Sweden pipeline:
- 10-min feature frames with 5-min hop
- timestamp alignment to the 5-min grid
- 120-min superwindows, 40-min stride
- 23 frames per superwindow
- per-feature 80% observed-frame QC across dynamic features
- static GA/sex/birthweight filled across the window
- per-window per-feature linear interpolation after per-feature QC
- mask channel: 1 where the value was originally missing, otherwise 0

Important external-validation choices
-------------------------------------
- LOS, abdominal NEC, sepsis and other adverse patients are RETAINED in every
  output, exactly as in the old usa_data notebook. Their labels are metadata
  only and do not create separate cohort files.
- PMA is derived as GA + PNA_days / 7.
- PMA50_PATIENT_CUT removes the entire patient if any finite source PMA is
  below 0 or above 50 weeks.
- PMA50_WINDOW_CUT retains the patient but removes only model windows whose
  target PMA is below 0 or above 50 weeks.
- All valid GA values are processed once; GA and PMA cohort rules are applied
  during final consolidation, so the expensive window conversion is shared.
- PNA, Apgar and adverse variables are never model inputs.
- No USA-derived normalization is performed here. Frozen Sweden checkpoint
  normalization must be applied during inference.
"""

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import pickle
import shutil
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


# ============================================================================
# 0. USER CONFIGURATION
# ============================================================================
PROJECT_ROOT = Path("/mnt/home/qinqiu/thesis")

USA_PATIENT_DIR = (
    PROJECT_ROOT / "data" / "US_patient_features" / "patients"
)

# This is the common-feature Sweden PKL used by the completed Optuna run.
SWEDEN_COMMON_PKL = (
    PROJECT_ROOT
    / "data"
    / "sweden_usa_common_feature_120min_stride40min"
    / "sweden_120min_stride40min_usa_common17_imputed_2ch_CTHW.pkl"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "USA_common_feature_120min_stride40min_external"
)

FINAL_OUTPUT_ALL_GA_PMA50_PATIENT_CUT_PKL = (
    OUTPUT_DIR
    / "usa_common17_120min_stride40min_ALL_GA_PMA50_PATIENT_CUT_imputed_2ch_CTHW.pkl"
)

FINAL_OUTPUT_ALL_GA_PMA50_WINDOW_CUT_PKL = (
    OUTPUT_DIR
    / "usa_common17_120min_stride40min_ALL_GA_PMA50_WINDOW_CUT_imputed_2ch_CTHW.pkl"
)

FINAL_OUTPUT_GA_LE35_PMA50_PATIENT_CUT_PKL = (
    OUTPUT_DIR
    / "usa_common17_120min_stride40min_GA_LE35_PMA50_PATIENT_CUT_imputed_2ch_CTHW.pkl"
)

FINAL_OUTPUT_GA_LE35_PMA50_WINDOW_CUT_PKL = (
    OUTPUT_DIR
    / "usa_common17_120min_stride40min_GA_LE35_PMA50_WINDOW_CUT_imputed_2ch_CTHW.pkl"
)

# Per-patient checkpoint files make the 2,938-patient conversion resumable.
PATIENT_CACHE_DIR = OUTPUT_DIR / "patient_conversion_cache"
LOG_DIR = OUTPUT_DIR / "logs"

# Run only the first N patients for a smoke test. Use None for the formal run.
MAX_PATIENTS: Optional[int] = None

# If True, valid existing patient caches are reused.
RESUME_PATIENT_CACHE = True

# If True, rebuild every patient even if a valid cache exists.
FORCE_REBUILD_PATIENT_CACHE = False

# Replace the four previously generated PKLs, because their QC rule was incorrect.
# After one successful formal run, this can be changed back to False for safety.
OVERWRITE_FINAL_PKL = True

# Process all physiologically valid GA values once. Cohort restriction is applied
# only during final consolidation, so no patient needs to be re-segmented.
MIN_GA_WEEKS = 0.0
PRIMARY_MAX_GA_WEEKS = 35.0

# PMA cut thresholds used by both final cohort variants.
MIN_PMA_WEEKS = 0.0
MAX_PMA_WEEKS = 50.0

# Incremented because QC now checks the observed-frame ratio separately for each
# dynamic feature. This invalidates caches built with the overall-window QC.
PIPELINE_VERSION = "usa_external_ga_pma4_v7_per_feature80qc_timestamp_resolution_fix_2026-08-03"

# Adverse infants must remain in external validation.
KEEP_ADVERSE_PATIENTS = True

# A model prediction requires all three static covariates.
REQUIRE_ALL_STATIC_FEATURES = True

# Store full per-window adverse/time-to-event metadata in the final PKL.
# This increases file size but keeps all external-analysis information together.
INCLUDE_FULL_WINDOW_ADVERSE_METADATA_IN_FINAL_PKL = True

# Sweden's final imputed PKL contains both X_by_patient and X2_by_patient.
# Keep both here for downstream structural compatibility, despite the larger file.
STORE_RAW_X_BY_PATIENT = True

# Progress reporting.
PRINT_EVERY_N_PATIENTS = 25

# Print one diagnostic line for every patient removed before final GA/PMA cohort cuts.
# Set to False to silence these lines. Use an integer limit (for example 100) to
# print only the first N removed patients; None prints all removed patients.
PRINT_REMOVED_PATIENT_DETAILS = True
PRINT_REMOVED_PATIENT_LIMIT: Optional[int] = None

# Strict expected temporal contract for the completed common CNN.
EXPECTED_FRAME_LEN_MIN = 10
EXPECTED_FRAME_HOP_MIN = 5
EXPECTED_SUPERWINDOW_MIN = 120
EXPECTED_SUPER_STRIDE_MIN = 40
EXPECTED_FRAMES_PER_SUPER = 23

# Per-feature QC threshold: every dynamic feature must have at least 80%
# observed frames in the 23-frame superwindow. Equivalently, if any feature
# has >20% missing frames, the entire window is rejected. Static features
# are excluded; remaining missing cells in retained windows are interpolated.
FALLBACK_VALID_RATIO = 0.80


COMMON_DYNAMIC_FEATURES = [
    "feats__spo2_mean",
    "feats__spo2_std",
    "feats__spo2_max",
    "feats__spo2_min",
    "feats__spo2_skew",
    "feats__spo2_kurtosis",
    "feats__btb_mean",
    "feats__btb_std",
    "feats__btb_max",
    "feats__btb_min",
    "feats__btb_skew",
    "feats__btb_kurtosis",
    "feats__btb_sampAs",
    "feats__btb_sampEn",
]

STATIC_FEATURES = [
    "feats__ga_w",
    "feats__sex",
    "feats__bw",
]

EXPECTED_FEATURE_ORDER = COMMON_DYNAMIC_FEATURES + STATIC_FEATURES
PNA_COLUMN = "feats__pna_days"
ID_COLUMN = "ids__uid"
TIMESTAMP_COLUMN = "timestamp"
SWEDEN_TARGET_COLUMN = "target__pma_w"
TRUE_PMA_COLUMN = "target__pma_w_external"

# These are retained only as metadata and can never enter model features.
FORBIDDEN_MODEL_FEATURES = [
    "feats__pna_days",
    "feats__apgar_1",
    "feats__apgar_5",
    "feats__weight",
]


# ============================================================================
# 1. GENERAL UTILITIES
# ============================================================================
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def stable_hash(payload: Any) -> str:
    encoded = json.dumps(
        json_safe(payload),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def atomic_pickle_dump(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)


def atomic_json_dump(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(json_safe(payload), f, indent=2, sort_keys=True)
    os.replace(tmp, path)


def first_finite_numeric(series: pd.Series) -> float:
    x = pd.to_numeric(series, errors="coerce")
    x = x[np.isfinite(x.to_numpy(dtype=float, na_value=np.nan))]
    if len(x) == 0:
        return np.nan
    return float(x.iloc[0])


def first_nonmissing(series: pd.Series) -> Any:
    x = series.dropna()
    if len(x) == 0:
        return None
    v = x.iloc[0]
    return v.item() if hasattr(v, "item") else v


def numeric_max_or_nan(series: pd.Series) -> float:
    x = pd.to_numeric(series, errors="coerce")
    if not x.notna().any():
        return np.nan
    return float(x.max(skipna=True))


def numeric_min_or_nan(series: pd.Series) -> float:
    x = pd.to_numeric(series, errors="coerce")
    if not x.notna().any():
        return np.nan
    return float(x.min(skipna=True))


def safe_file_identity(path: Path) -> Dict[str, Any]:
    stat = path.stat()
    return {
        "path": str(path.resolve()),
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def read_json_if_available(path: Path) -> Optional[Dict[str, Any]]:
    if not path.is_file():
        return None
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if not isinstance(obj, dict):
        raise ValueError(f"Expected JSON object in {path}, got {type(obj)}")
    return obj


def require_parquet_engine() -> None:
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            "PyArrow is required. In the same Jupyter environment run:\n"
            "    %pip install pyarrow\n"
            "then restart the kernel."
        ) from exc


# ============================================================================
# 2. LOAD AND VERIFY THE SWEDEN INPUT CONTRACT
# ============================================================================
def load_sweden_contract(sweden_pkl: Path) -> Dict[str, Any]:
    if not sweden_pkl.is_file():
        raise FileNotFoundError(f"Sweden common PKL not found: {sweden_pkl}")

    with open(sweden_pkl, "rb") as f:
        sweden = pickle.load(f)

    feature_cols = list(sweden.get("feature_cols", []))
    if feature_cols != EXPECTED_FEATURE_ORDER:
        raise ValueError(
            "Sweden common feature order does not match the expected 17-column "
            "contract.\n"
            f"Expected: {EXPECTED_FEATURE_ORDER}\n"
            f"Found:    {feature_cols}"
        )

    params = dict(sweden.get("params", {}))

    frame_len = int(params.get("frame_len_minutes", EXPECTED_FRAME_LEN_MIN))
    frame_hop = int(params.get("frame_hop_minutes", EXPECTED_FRAME_HOP_MIN))
    superwindow = int(
        params.get("superwindow_minutes", EXPECTED_SUPERWINDOW_MIN)
    )
    stride = int(
        params.get("super_stride_minutes", EXPECTED_SUPER_STRIDE_MIN)
    )
    frames_per_super = int(
        params.get(
            "frames_per_super",
            int((superwindow - frame_len) / frame_hop) + 1,
        )
    )

    fallback_min_valid = max(
        1,
        math.floor(frames_per_super * FALLBACK_VALID_RATIO + 1e-9),
    )
    min_valid = int(
        params.get("min_valid_frames_per_feature", fallback_min_valid)
    )

    expected_temporal = {
        "frame_len_minutes": EXPECTED_FRAME_LEN_MIN,
        "frame_hop_minutes": EXPECTED_FRAME_HOP_MIN,
        "superwindow_minutes": EXPECTED_SUPERWINDOW_MIN,
        "super_stride_minutes": EXPECTED_SUPER_STRIDE_MIN,
        "frames_per_super": EXPECTED_FRAMES_PER_SUPER,
    }
    actual_temporal = {
        "frame_len_minutes": frame_len,
        "frame_hop_minutes": frame_hop,
        "superwindow_minutes": superwindow,
        "super_stride_minutes": stride,
        "frames_per_super": frames_per_super,
    }

    if actual_temporal != expected_temporal:
        raise ValueError(
            "The Sweden common PKL temporal contract differs from the completed "
            "120-min/40-min model.\n"
            f"Expected: {expected_temporal}\n"
            f"Found:    {actual_temporal}"
        )

    if not (1 <= min_valid <= frames_per_super):
        raise ValueError(
            f"Invalid min_valid_frames_per_feature={min_valid}; "
            f"frames_per_super={frames_per_super}"
        )

    # Confirm example X shape when available.
    example_shape = None
    for _, x in sweden.get("X2_by_patient", {}).items():
        if x is not None and np.asarray(x).ndim == 4 and len(x) > 0:
            example_shape = tuple(np.asarray(x).shape)
            break

    if example_shape is not None:
        if example_shape[1:] != (2, frames_per_super, len(feature_cols)):
            raise ValueError(
                "Sweden example X2 shape is incompatible with the contract: "
                f"{example_shape}"
            )

    source_identity = safe_file_identity(sweden_pkl)
    contract = {
        **actual_temporal,
        "min_valid_frames_per_feature": min_valid,
        "feature_cols": feature_cols,
        "dynamic_feature_cols": list(COMMON_DYNAMIC_FEATURES),
        "static_feature_cols": list(STATIC_FEATURES),
        "qc_ignore_static": bool(params.get("qc_ignore_static", True)),
        "align_timestamps": bool(params.get("align_timestamps", True)),
        "align_method": str(params.get("align_method", "round")),
        "target_aggregation": "mean_over_observed_frames_on_fixed_grid",
        "imputation": (
            "per-window per-feature linear interpolation; nearest edge fill; "
            "mask=1 for original NaN"
        ),
        "sweden_common_pkl": str(sweden_pkl.resolve()),
        "sweden_common_pkl_identity": source_identity,
        "sweden_example_X2_shape": example_shape,
    }
    contract["contract_hash"] = stable_hash(contract)
    return contract


# ============================================================================
# 3. TIMESTAMP, IMPUTATION AND METADATA HELPERS
# ============================================================================
def align_to_hop_grid(
    ts: pd.Series,
    frame_hop_minutes: int,
    align_method: str,
) -> pd.Series:
    """Align timestamps to the hop grid without assuming datetime integer units.

    Parquet-backed DatetimeIndex values may use datetime64[us] rather than
    datetime64[ns]. Converting them with ``astype("int64")`` and then dividing
    by a nanosecond constant compresses the time axis by 1000. Pandas' native
    datetime rounding is resolution-safe and preserves the original 5-minute
    sequence.
    """
    parsed = pd.to_datetime(ts, errors="coerce")
    if parsed.isna().all():
        raise ValueError("All timestamps became NaT.")

    h = int(frame_hop_minutes)
    if h <= 0:
        raise ValueError("frame_hop_minutes must be positive.")
    freq = f"{h}min"

    if isinstance(parsed, pd.Series):
        if align_method == "round":
            aligned = parsed.dt.round(freq)
        elif align_method == "floor":
            aligned = parsed.dt.floor(freq)
        else:
            raise ValueError("align_method must be 'round' or 'floor'.")

        return pd.Series(
            aligned.to_numpy(),
            index=ts.index,
            name=getattr(ts, "name", None),
        )

    # Defensive support if a DatetimeIndex is supplied directly.
    if align_method == "round":
        aligned = parsed.round(freq)
    elif align_method == "floor":
        aligned = parsed.floor(freq)
    else:
        raise ValueError("align_method must be 'round' or 'floor'.")

    return pd.Series(aligned.to_numpy(), name=getattr(ts, "name", None))


def impute_linear_time_with_mask(
    x_tf: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Replicate the Sweden per-window imputation.

    Parameters
    ----------
    x_tf : (T, F) float array with NaNs

    Returns
    -------
    x_imputed_tf : (T, F) float32, no NaNs
    mask_tf      : (T, F) float32, 1 where original was NaN, else 0
    """
    if x_tf.ndim != 2:
        raise ValueError(f"Expected (T,F), got {x_tf.shape}")

    t_size, f_size = x_tf.shape
    mask_tf = np.isnan(x_tf).astype(np.float32)
    x_imputed = x_tf.astype(np.float32, copy=True)
    t_idx = np.arange(t_size, dtype=np.float32)

    for f_idx in range(f_size):
        col = x_imputed[:, f_idx]
        nan = np.isnan(col)

        if not nan.any():
            continue

        valid = ~nan
        if valid.sum() == 0:
            col[:] = 0.0
            x_imputed[:, f_idx] = col
            continue

        col[nan] = np.interp(
            t_idx[nan],
            t_idx[valid],
            col[valid],
        ).astype(np.float32)

        if np.isnan(col).any():
            mean_value = np.nanmean(col)
            col[np.isnan(col)] = (
                0.0 if not np.isfinite(mean_value) else np.float32(mean_value)
            )

        x_imputed[:, f_idx] = col

    return x_imputed, mask_tf


def get_timestamp_series(df: pd.DataFrame) -> pd.Series:
    if TIMESTAMP_COLUMN in df.columns:
        return pd.to_datetime(df[TIMESTAMP_COLUMN], errors="coerce")

    if isinstance(df.index, pd.DatetimeIndex):
        return pd.Series(
            pd.to_datetime(df.index, errors="coerce"),
            index=df.index,
            name=TIMESTAMP_COLUMN,
        )

    if df.index.name == TIMESTAMP_COLUMN:
        return pd.Series(
            pd.to_datetime(df.index, errors="coerce"),
            index=df.index,
            name=TIMESTAMP_COLUMN,
        )

    raise ValueError(
        f"No '{TIMESTAMP_COLUMN}' column and index is not a DatetimeIndex."
    )


def json_feature_contract(cache_json: Optional[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    if cache_json is None:
        return None
    extraction = cache_json.get("extraction_contract", {})
    return {
        "version": cache_json.get("version"),
        "columns": cache_json.get("columns"),
        "index_name": cache_json.get("index_name"),
        "causal_interpolation": extraction.get("causal_interpolation"),
        "feats": extraction.get("feats"),
        "signal": extraction.get("signal"),
        "window_feature_mode": extraction.get("window_feature_mode"),
    }


def get_static_values(
    df: pd.DataFrame,
    static_cols: Sequence[str],
) -> Tuple[Dict[str, float], Dict[str, int]]:
    values: Dict[str, float] = {}
    unique_counts: Dict[str, int] = {}

    for col in static_cols:
        if col not in df.columns:
            values[col] = np.nan
            unique_counts[col] = 0
            continue

        numeric = pd.to_numeric(df[col], errors="coerce")
        non_na = numeric.dropna()
        unique_counts[col] = int(non_na.nunique(dropna=True))
        values[col] = float(non_na.iloc[0]) if len(non_na) else np.nan

    return values, unique_counts


def build_patient_metadata(
    df: pd.DataFrame,
    cache_json: Optional[Dict[str, Any]],
    pid: str,
    parquet_path: Path,
    json_path: Path,
    static_values: Dict[str, float],
    static_unique_counts: Dict[str, int],
    aligned_time: pd.Series,
) -> Dict[str, Any]:
    meta: Dict[str, Any] = {
        "pid": pid,
        "source_parquet": str(parquet_path),
        "source_json": str(json_path) if json_path.is_file() else None,
        "n_source_rows": int(len(df)),
        "aligned_start_time": aligned_time.min(),
        "aligned_end_time": aligned_time.max(),
        "static_values": dict(static_values),
        "static_unique_counts": dict(static_unique_counts),
        "json_extraction_fingerprint": (
            cache_json.get("extraction_fingerprint")
            if cache_json is not None
            else None
        ),
        "json_feature_sha256": (
            cache_json.get("feature_sha256")
            if cache_json is not None
            else None
        ),
        "json_manifest": (
            cache_json.get("extraction_contract", {}).get("manifest")
            if cache_json is not None
            else None
        ),
    }

    # Patient-level labels and event ages already repeated in each parquet row.
    patient_cols = [c for c in df.columns if str(c).startswith("patient_")]
    for col in patient_cols:
        if col.startswith("patient_event__") or col.startswith("patient_has_"):
            meta[col] = numeric_max_or_nan(df[col])
        elif col.startswith("patient_first_event_age_hours__"):
            meta[col] = first_finite_numeric(df[col])
        elif col == "patient_exclude_from_control":
            meta[col] = numeric_max_or_nan(df[col])
        else:
            meta[col] = first_nonmissing(df[col])

    # Summarize every target at patient level as "ever positive" / max.
    target_cols = [c for c in df.columns if str(c).startswith("target__")]
    for col in target_cols:
        meta[f"patient_max__{col}"] = numeric_max_or_nan(df[col])

    # Convenience external-analysis flags.
    adverse_candidates = [
        "target__neo_adverse",
        "target__sepsis",
        "target__los",
        "target__abdominal_nec",
        "target__death",
        "target__infection",
        "target__bleeding",
        "target__brain_ivh_stage_3_4",
    ]
    adverse_values = []
    for col in adverse_candidates:
        if col in df.columns:
            v = numeric_max_or_nan(df[col])
            if np.isfinite(v):
                adverse_values.append(v)

    meta["patient_any_adverse"] = (
        int(any(v > 0 for v in adverse_values))
        if adverse_values
        else None
    )

    exclude_control = meta.get("patient_exclude_from_control")
    meta["patient_strict_control_candidate"] = (
        int(np.isfinite(exclude_control) and float(exclude_control) == 0.0)
        if isinstance(exclude_control, (int, float, np.integer, np.floating))
        else None
    )

    # Explicit mutually exclusive cohort label. Both adverse and non-adverse
    # patients are retained; this is metadata only and never a model input.
    any_adverse = meta.get("patient_any_adverse")
    strict_control = meta.get("patient_strict_control_candidate")
    if any_adverse == 1:
        adverse_group = "adverse"
    elif any_adverse == 0 and strict_control == 1:
        adverse_group = "control"
    elif any_adverse == 0:
        adverse_group = "nonadverse_not_strict_control"
    else:
        adverse_group = "unknown"

    meta["adverse_group"] = adverse_group
    meta["ga_w"] = float(static_values.get("feats__ga_w", np.nan))
    meta["ga_le_35"] = int(meta["ga_w"] <= PRIMARY_MAX_GA_WEEKS)

    return meta


# ============================================================================
# 4. PROCESS ONE PATIENT
# ============================================================================
def process_patient_dataframe(
    df_input: pd.DataFrame,
    cache_json: Optional[Dict[str, Any]],
    pid: str,
    parquet_path: Path,
    json_path: Path,
    contract: Dict[str, Any],
) -> Dict[str, Any]:
    df = df_input.copy()

    required_cols = list(contract["feature_cols"]) + [PNA_COLUMN]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        return {
            "status": "removed",
            "reason": f"missing_required_columns:{missing_cols}",
            "pid": pid,
        }

    timestamp = get_timestamp_series(df)
    if timestamp.isna().all():
        return {
            "status": "removed",
            "reason": "all_timestamps_missing",
            "pid": pid,
        }

    # Work with a clean RangeIndex so timestamp alignment cannot be confused by
    # the original DatetimeIndex.
    df = df.reset_index(drop=True)
    timestamp = pd.Series(timestamp.to_numpy(), index=df.index)

    # Ensure all model/target columns are numeric.
    numeric_cols = list(contract["feature_cols"]) + [PNA_COLUMN]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    static_values, static_unique_counts = get_static_values(
        df,
        contract["static_feature_cols"],
    )

    ga_w = static_values["feats__ga_w"]
    if not np.isfinite(ga_w):
        return {"status": "removed", "reason": "missing_GA", "pid": pid}
    if ga_w < MIN_GA_WEEKS:
        return {"status": "removed", "reason": "GA_below_zero", "pid": pid}
    if REQUIRE_ALL_STATIC_FEATURES:
        missing_static = [
            c for c, v in static_values.items() if not np.isfinite(v)
        ]
        if missing_static:
            return {
                "status": "removed",
                "reason": f"missing_static:{missing_static}",
                "pid": pid,
            }

    # ---------------------------------------------------------------
    # PMA is measured here but filtering is deferred until consolidation.
    # This allows patient-level and window-level PMA cuts to share one cache.
    # ---------------------------------------------------------------
    source_pna = pd.to_numeric(df[PNA_COLUMN], errors="coerce")
    source_true_pma = ga_w + source_pna / 7.0
    finite_source_pma = source_true_pma[np.isfinite(source_true_pma)]

    if len(finite_source_pma) == 0:
        return {
            "status": "removed",
            "reason": "no_finite_PMA",
            "pid": pid,
        }

    source_pma_min = float(finite_source_pma.min())
    source_pma_max = float(finite_source_pma.max())

    aligned_time = align_to_hop_grid(
        timestamp,
        frame_hop_minutes=contract["frame_hop_minutes"],
        align_method=contract["align_method"],
    )

    df["_frame_time"] = aligned_time.to_numpy()
    df = df.dropna(subset=["_frame_time"]).copy()
    if len(df) == 0:
        return {
            "status": "removed",
            "reason": "no_rows_after_timestamp_alignment",
            "pid": pid,
        }

    n_before_dedup = int(len(df))
    df = (
        df.sort_values("_frame_time", kind="stable")
        .drop_duplicates(subset=["_frame_time"], keep="first")
        .set_index("_frame_time")
        .sort_index()
    )
    n_duplicate_aligned_rows = n_before_dedup - int(len(df))

    # Patient-level metadata is extracted before feature slicing.
    patient_meta = build_patient_metadata(
        df=df,
        cache_json=cache_json,
        pid=pid,
        parquet_path=parquet_path,
        json_path=json_path,
        static_values=static_values,
        static_unique_counts=static_unique_counts,
        aligned_time=pd.Series(df.index),
    )
    patient_meta["source_PMA_min"] = source_pma_min
    patient_meta["source_PMA_max"] = source_pma_max
    patient_meta["PMA_filter_during_conversion"] = "none_deferred_to_consolidation"
    patient_meta["source_PMA_outside_0_50"] = bool(
        source_pma_min < MIN_PMA_WEEKS or source_pma_max > MAX_PMA_WEEKS
    )

    feature_cols = list(contract["feature_cols"])
    dynamic_cols = list(contract["dynamic_feature_cols"])
    static_cols = list(contract["static_feature_cols"])

    feat_df = df[feature_cols].copy()
    for col in static_cols:
        feat_df[col] = static_values[col]

    # External ground truth. PNA remains a label-construction variable only.
    pna_series = pd.to_numeric(df[PNA_COLUMN], errors="coerce")
    true_pma_series = ga_w + pna_series / 7.0
    true_pma_series.name = TRUE_PMA_COLUMN

    # Dynamic adverse metadata retained for later event/subgroup analysis.
    target_cols = [c for c in df.columns if str(c).startswith("target__")]
    time_to_cols = [c for c in df.columns if str(c).startswith("time_to__")]
    for col in target_cols + time_to_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    t_min = feat_df.index.min()
    t_max = feat_df.index.max()
    stride = pd.Timedelta(minutes=contract["super_stride_minutes"])
    hop = pd.Timedelta(minutes=contract["frame_hop_minutes"])
    super_len = pd.Timedelta(minutes=contract["superwindow_minutes"])
    frames_per_super = int(contract["frames_per_super"])
    per_feature_valid_ratio_threshold = float(FALLBACK_VALID_RATIO)

    starts: List[pd.Timestamp] = []
    current = t_min
    while current <= t_max:
        starts.append(current)
        current = current + stride

    x2_blocks: List[np.ndarray] = []
    raw_blocks: List[np.ndarray] = []
    y_list: List[float] = []
    basic_meta_rows: List[Dict[str, Any]] = []
    adverse_meta_rows: List[Dict[str, Any]] = []

    rejected_qc = 0
    rejected_target = 0
    candidate_missing_fractions: List[float] = []

    center_idx = frames_per_super // 2

    for candidate_idx, start_time in enumerate(starts):
        grid = pd.date_range(
            start=start_time,
            periods=frames_per_super,
            freq=hop,
        )

        block_df = feat_df.reindex(grid)
        for col in static_cols:
            block_df[col] = static_values[col]

        # Per-feature QC: evaluate each dynamic feature separately over the
        # 23-frame superwindow. Static GA/sex/birthweight are excluded.
        dynamic_block = block_df[dynamic_cols]
        valid_counts = dynamic_block.notna().sum(axis=0)

        per_feature_valid_ratio = valid_counts.astype(float) / float(frames_per_super)
        per_feature_missing_fraction = 1.0 - per_feature_valid_ratio

        # Keep the existing audit-stat keys unchanged. For this QC version,
        # candidate_missing_fractions stores the WORST (maximum) feature-level
        # missing fraction for each candidate window.
        max_feature_missing_fraction = float(per_feature_missing_fraction.max())
        candidate_missing_fractions.append(max_feature_missing_fraction)

        # Strict interpretation of "missing >20%": reject if any dynamic
        # feature has a missing fraction strictly greater than 0.20.
        if (per_feature_missing_fraction > (1.0 - per_feature_valid_ratio_threshold)).any():
            rejected_qc += 1
            continue

        target_block = true_pma_series.reindex(grid).to_numpy(dtype=float)
        target_mean = float(np.nanmean(target_block))
        if not np.isfinite(target_mean):
            rejected_target += 1
            continue

        raw_tf = block_df[feature_cols].to_numpy(dtype=float)
        imputed_tf, mask_tf = impute_linear_time_with_mask(raw_tf)

        if not np.isfinite(imputed_tf).all():
            raise RuntimeError(f"pid={pid}: nonfinite values after imputation")
        if not np.isfinite(mask_tf).all():
            raise RuntimeError(f"pid={pid}: nonfinite mask values")

        x2 = np.stack([imputed_tf, mask_tf], axis=0).astype(np.float32)
        expected_shape = (
            2,
            frames_per_super,
            len(feature_cols),
        )
        if x2.shape != expected_shape:
            raise RuntimeError(
                f"pid={pid}: X2 block shape={x2.shape}, expected={expected_shape}"
            )

        final_window_idx = len(x2_blocks)
        x2_blocks.append(x2)
        if STORE_RAW_X_BY_PATIENT:
            raw_blocks.append(raw_tf.astype(np.float32))
        y_list.append(target_mean)

        pna_block = pna_series.reindex(grid)
        center_time = grid[center_idx]
        basic_meta_row = {
                # Sweden-compatible names
                "super_id": final_window_idx,
                "start_time": start_time,
                "end_time": start_time + super_len,
                f"{SWEDEN_TARGET_COLUMN}_mean": target_mean,
                # USA audit aliases / additional metadata
                "window_idx": final_window_idx,
                "candidate_window_idx": candidate_idx,
                "center_time": center_time,
                SWEDEN_TARGET_COLUMN: target_mean,
                TRUE_PMA_COLUMN: target_mean,
                "PMA_window_in_range_0_50": bool(
                    MIN_PMA_WEEKS <= target_mean <= MAX_PMA_WEEKS
                ),
                "pna_days_mean": float(pna_block.mean(skipna=True)),
                "pna_days_center": (
                    float(pna_block.iloc[center_idx])
                    if pd.notna(pna_block.iloc[center_idx])
                    else np.nan
                ),
                "n_observed_grid_rows": int(
                    block_df[dynamic_cols].notna().any(axis=1).sum()
                ),
                "dynamic_missing_fraction": float(
                    block_df[dynamic_cols].isna().to_numpy().mean()
                ),
                "min_dynamic_valid_frames": int(valid_counts.min()),
                "max_dynamic_valid_frames": int(valid_counts.max()),
            }
        for col in static_cols:
            basic_meta_row[col] = static_values[col]
        basic_meta_rows.append(basic_meta_row)

        adverse_row: Dict[str, Any] = {
            "window_idx": final_window_idx,
            "start_time": start_time,
            "end_time": start_time + super_len,
            "center_time": center_time,
        }

        # For a binary/horizon target, max captures whether it is active anywhere
        # in the 120-min window; center preserves the exact center-frame label.
        for col in target_cols:
            series_block = df[col].reindex(grid)
            adverse_row[f"{col}__window_max"] = (
                float(series_block.max(skipna=True))
                if series_block.notna().any()
                else np.nan
            )
            center_value = series_block.iloc[center_idx]
            adverse_row[f"{col}__center"] = (
                float(center_value) if pd.notna(center_value) else np.nan
            )

        # time_to variables are evaluated at the center feature frame.
        for col in time_to_cols:
            series_block = df[col].reindex(grid)
            center_value = series_block.iloc[center_idx]
            adverse_row[f"{col}__center"] = (
                float(center_value) if pd.notna(center_value) else np.nan
            )

        adverse_meta_rows.append(adverse_row)

    n_kept = len(x2_blocks)
    if n_kept == 0:
        return {
            "status": "removed",
            "reason": "zero_windows_after_QC",
            "pid": pid,
            "patient_metadata": patient_meta,
            "stats": {
                "n_source_rows": int(len(df_input)),
                "n_aligned_unique_rows": int(len(df)),
                "n_duplicate_aligned_rows": int(n_duplicate_aligned_rows),
                "n_candidate_windows": int(len(starts)),
                "n_rejected_qc": int(rejected_qc),
                "n_rejected_target": int(rejected_target),
                "n_kept_windows": 0,
                "window_missing_fraction_min": (
                    float(np.min(candidate_missing_fractions))
                    if candidate_missing_fractions else np.nan
                ),
                "window_missing_fraction_median": (
                    float(np.median(candidate_missing_fractions))
                    if candidate_missing_fractions else np.nan
                ),
                "window_missing_fraction_max": (
                    float(np.max(candidate_missing_fractions))
                    if candidate_missing_fractions else np.nan
                ),
                "window_missing_fraction_threshold": float(
                    1.0 - per_feature_valid_ratio_threshold
                ),
                "n_windows_passing_qc_before_target": int(
                    len(starts) - rejected_qc
                ),
                "source_PMA_min": source_pma_min,
                "source_PMA_max": source_pma_max,
            },
        }

    x2_patient = np.stack(x2_blocks, axis=0).astype(np.float32)
    y_patient = np.asarray(y_list, dtype=np.float32)

    expected_patient_tail = (
        2,
        frames_per_super,
        len(feature_cols),
    )
    if x2_patient.shape[1:] != expected_patient_tail:
        raise RuntimeError(
            f"pid={pid}: final X2 tail={x2_patient.shape[1:]}, "
            f"expected={expected_patient_tail}"
        )
    if len(x2_patient) != len(y_patient):
        raise RuntimeError(f"pid={pid}: X/y length mismatch")

    payload: Dict[str, Any] = {
        "status": "kept",
        "pid": pid,
        "X2": x2_patient,
        "y": y_patient,
        "meta": pd.DataFrame(basic_meta_rows),
        "patient_metadata": patient_meta,
        "window_adverse_metadata": pd.DataFrame(adverse_meta_rows),
        "stats": {
            "n_source_rows": int(len(df_input)),
            "n_aligned_unique_rows": int(len(df)),
            "n_duplicate_aligned_rows": int(n_duplicate_aligned_rows),
            "n_candidate_windows": int(len(starts)),
            "n_rejected_qc": int(rejected_qc),
            "n_rejected_target": int(rejected_target),
            "n_kept_windows": int(n_kept),
            "window_missing_fraction_min": (
                float(np.min(candidate_missing_fractions))
                if candidate_missing_fractions else np.nan
            ),
            "window_missing_fraction_median": (
                float(np.median(candidate_missing_fractions))
                if candidate_missing_fractions else np.nan
            ),
            "window_missing_fraction_max": (
                float(np.max(candidate_missing_fractions))
                if candidate_missing_fractions else np.nan
            ),
            "window_missing_fraction_threshold": float(
                1.0 - per_feature_valid_ratio_threshold
            ),
            "n_windows_passing_qc_before_target": int(
                len(starts) - rejected_qc
            ),
            "source_PMA_min": source_pma_min,
            "source_PMA_max": source_pma_max,
            "retention_ratio": (
                float(n_kept / len(starts)) if len(starts) else np.nan
            ),
            "total_missing_values_before_imputation": int(
                sum(int(x[1].sum()) for x in x2_blocks)
            ),
        },
    }

    if STORE_RAW_X_BY_PATIENT:
        payload["X_raw"] = np.stack(raw_blocks, axis=0).astype(np.float32)

    return payload


# ============================================================================
# 5. PER-PATIENT RESUMABLE CONVERSION
# ============================================================================
def patient_cache_path(pid: str) -> Path:
    safe_pid = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in pid)
    return PATIENT_CACHE_DIR / f"{safe_pid}.pkl"


def make_patient_cache_identity(
    parquet_path: Path,
    json_path: Path,
    contract: Dict[str, Any],
) -> Dict[str, Any]:
    identity = {
        "pipeline_version": PIPELINE_VERSION,
        "parquet": safe_file_identity(parquet_path),
        "json": safe_file_identity(json_path) if json_path.is_file() else None,
        "contract_hash": contract["contract_hash"],
        "ga_min": MIN_GA_WEEKS,
        "ga_upper_filter_during_conversion": None,
        "primary_ga_threshold": PRIMARY_MAX_GA_WEEKS,
        "pma_min": MIN_PMA_WEEKS,
        "pma_max": MAX_PMA_WEEKS,
        "pma_filter_level": "deferred_to_final_consolidation",
        "keep_adverse": KEEP_ADVERSE_PATIENTS,
        "require_all_static": REQUIRE_ALL_STATIC_FEATURES,
        "store_raw": STORE_RAW_X_BY_PATIENT,
    }
    identity["identity_hash"] = stable_hash(identity)
    return identity


def load_valid_patient_cache(
    cache_path: Path,
    expected_identity: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    if not cache_path.is_file():
        return None
    try:
        with open(cache_path, "rb") as f:
            cached = pickle.load(f)
    except Exception:
        return None

    if cached.get("cache_identity") != expected_identity:
        return None
    if "result" not in cached:
        return None
    return cached["result"]


def convert_one_patient_file(
    parquet_path: Path,
    contract: Dict[str, Any],
) -> Tuple[Dict[str, Any], bool]:
    pid = str(parquet_path.stem)
    json_path = parquet_path.with_name(parquet_path.name + ".cache.json")
    cache_path = patient_cache_path(pid)
    cache_identity = make_patient_cache_identity(
        parquet_path,
        json_path,
        contract,
    )

    if (
        RESUME_PATIENT_CACHE
        and not FORCE_REBUILD_PATIENT_CACHE
        and cache_path.is_file()
    ):
        result = load_valid_patient_cache(cache_path, cache_identity)
        if result is not None:
            return result, True

    cache_json = read_json_if_available(json_path)
    df = pd.read_parquet(parquet_path, engine="pyarrow")

    # Audit that filename and parquet UID agree when UID is available.
    if ID_COLUMN in df.columns:
        uid_values = pd.Series(df[ID_COLUMN]).dropna().astype(str).unique()
        if len(uid_values) > 1:
            raise ValueError(
                f"{parquet_path}: multiple {ID_COLUMN} values: {uid_values[:10]}"
            )
        if len(uid_values) == 1 and uid_values[0] != pid:
            # Numeric parquet values may become '1000600.0'; normalize when safe.
            try:
                uid_normalized = str(int(float(uid_values[0])))
            except Exception:
                uid_normalized = uid_values[0]
            if uid_normalized != pid:
                raise ValueError(
                    f"Filename PID={pid} but parquet {ID_COLUMN}={uid_values[0]}"
                )

    result = process_patient_dataframe(
        df_input=df,
        cache_json=cache_json,
        pid=pid,
        parquet_path=parquet_path,
        json_path=json_path,
        contract=contract,
    )

    atomic_pickle_dump(
        {
            "cache_identity": cache_identity,
            "result": result,
        },
        cache_path,
    )

    del df
    gc.collect()
    return result, False


# ============================================================================
# 6. FINAL CONSOLIDATION
# ============================================================================
def flatten_patient_metadata_for_csv(
    metadata_by_patient: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    rows = []
    for pid, meta in metadata_by_patient.items():
        row: Dict[str, Any] = {"pid": pid}
        for key, value in meta.items():
            if key == "pid":
                continue
            if isinstance(value, (dict, list, tuple)):
                row[key] = json.dumps(json_safe(value), sort_keys=True)
            elif isinstance(value, pd.Timestamp):
                row[key] = value.isoformat()
            else:
                row[key] = value
        rows.append(row)
    return pd.DataFrame(rows)


def consolidate_results(
    patient_files: Sequence[Path],
    contract: Dict[str, Any],
    cohort_name: str,
    max_ga_weeks: Optional[float],
    pma_filter_level: str,
) -> Dict[str, Any]:
    """Create one final cohort from the shared patient-window cache.

    pma_filter_level="patient": remove the whole patient when any finite source
    PMA is outside [MIN_PMA_WEEKS, MAX_PMA_WEEKS].

    pma_filter_level="window": retain the patient and remove only windows whose
    target PMA is outside that interval. A patient is removed only if no windows
    remain afterward.

    LOS/NEC and every other adverse group are retained as metadata in both modes.
    """
    if pma_filter_level not in {"patient", "window"}:
        raise ValueError(
            f"pma_filter_level must be 'patient' or 'window', got {pma_filter_level!r}"
        )

    x2_by_patient: Dict[str, np.ndarray] = {}
    y_by_patient: Dict[str, np.ndarray] = {}
    x_raw_by_patient: Dict[str, np.ndarray] = {}
    meta_by_patient: Dict[str, pd.DataFrame] = {}
    patient_metadata_by_patient: Dict[str, Dict[str, Any]] = {}
    window_adverse_metadata_by_patient: Dict[str, pd.DataFrame] = {}

    stats_rows = []
    removed_rows = []
    json_contract_hashes = Counter()
    total_windows_removed_by_pma = 0
    patients_retained_after_window_cut_despite_source_outlier = 0

    expected_tail = (
        2,
        contract["frames_per_super"],
        len(contract["feature_cols"]),
    )

    for parquet_path in patient_files:
        pid = str(parquet_path.stem)
        cache_path = patient_cache_path(pid)
        if not cache_path.is_file():
            raise FileNotFoundError(f"Missing patient cache: {cache_path}")

        with open(cache_path, "rb") as f:
            cached = pickle.load(f)
        result = cached["result"]

        status = result.get("status")
        stats = dict(result.get("stats", {}))
        stat_row = {
            "pid": pid,
            "patient_id": pid,
            "status": status,
            "reason": result.get("reason"),
            "n_super_before": stats.get("n_candidate_windows"),
            "n_super_after": stats.get("n_kept_windows"),
            **stats,
        }

        patient_meta = result.get("patient_metadata")

        if status != "kept":
            stats_rows.append(stat_row)
            removed_rows.append(
                {"pid": pid, "reason": result.get("reason"), **stats}
            )
            continue

        if not isinstance(patient_meta, dict):
            raise RuntimeError(f"pid={pid}: kept patient has no metadata")
        patient_meta = dict(patient_meta)

        ga_value = float(patient_meta.get("ga_w", np.nan))
        if max_ga_weeks is not None and (
            not np.isfinite(ga_value) or ga_value > float(max_ga_weeks)
        ):
            reason = f"GA_above_{float(max_ga_weeks):g}"
            stat_row.update(status="excluded_from_cohort", reason=reason)
            stats_rows.append(stat_row)
            removed_rows.append({"pid": pid, "reason": reason, **stats})
            continue

        x2 = np.asarray(result["X2"], dtype=np.float32)
        y = np.asarray(result["y"], dtype=np.float32)
        meta_df = result["meta"].copy().reset_index(drop=True)
        adverse_df = result["window_adverse_metadata"].copy().reset_index(drop=True)
        x_raw = (
            np.asarray(result["X_raw"], dtype=np.float32)
            if STORE_RAW_X_BY_PATIENT
            else None
        )

        if x2.shape[0] != len(y):
            raise RuntimeError(f"pid={pid}: cached X/y mismatch")
        if x2.shape[1:] != expected_tail:
            raise RuntimeError(f"pid={pid}: cached X2 shape mismatch {x2.shape}")
        if len(meta_df) != len(y) or len(adverse_df) != len(y):
            raise RuntimeError(f"pid={pid}: cached metadata/window mismatch")
        if x_raw is not None and len(x_raw) != len(y):
            raise RuntimeError(f"pid={pid}: cached raw-X/window mismatch")
        if not np.isfinite(x2).all() or not np.isfinite(y).all():
            raise RuntimeError(f"pid={pid}: nonfinite cached model data")

        source_pma_min = float(
            patient_meta.get("source_PMA_min", stats.get("source_PMA_min", np.nan))
        )
        source_pma_max = float(
            patient_meta.get("source_PMA_max", stats.get("source_PMA_max", np.nan))
        )
        source_outside = (
            not np.isfinite(source_pma_min)
            or not np.isfinite(source_pma_max)
            or source_pma_min < MIN_PMA_WEEKS
            or source_pma_max > MAX_PMA_WEEKS
        )

        n_windows_before_pma = int(len(y))
        n_windows_removed_pma = 0

        if pma_filter_level == "patient":
            if source_outside:
                reason = (
                    f"PMA_outside_{MIN_PMA_WEEKS:g}_{MAX_PMA_WEEKS:g}_patient_cut"
                )
                stat_row.update(
                    status="excluded_from_cohort",
                    reason=reason,
                    n_windows_before_pma_cut=n_windows_before_pma,
                    n_windows_removed_by_pma_cut=n_windows_before_pma,
                    n_windows_after_pma_cut=0,
                )
                stats_rows.append(stat_row)
                removed_rows.append(
                    {
                        "pid": pid,
                        "reason": reason,
                        "source_PMA_min": source_pma_min,
                        "source_PMA_max": source_pma_max,
                        **stats,
                    }
                )
                total_windows_removed_by_pma += n_windows_before_pma
                continue
            keep_mask = np.ones(len(y), dtype=bool)
        else:
            keep_mask = (
                np.isfinite(y)
                & (y >= MIN_PMA_WEEKS)
                & (y <= MAX_PMA_WEEKS)
            )
            n_windows_removed_pma = int((~keep_mask).sum())
            total_windows_removed_by_pma += n_windows_removed_pma
            if not keep_mask.any():
                reason = "zero_windows_after_PMA_window_cut"
                stat_row.update(
                    status="excluded_from_cohort",
                    reason=reason,
                    n_windows_before_pma_cut=n_windows_before_pma,
                    n_windows_removed_by_pma_cut=n_windows_removed_pma,
                    n_windows_after_pma_cut=0,
                )
                stats_rows.append(stat_row)
                removed_rows.append(
                    {
                        "pid": pid,
                        "reason": reason,
                        "source_PMA_min": source_pma_min,
                        "source_PMA_max": source_pma_max,
                        **stats,
                    }
                )
                continue
            if source_outside:
                patients_retained_after_window_cut_despite_source_outlier += 1

        # Apply the same selected-window mask to every per-window object.
        x2 = x2[keep_mask]
        y = y[keep_mask]
        meta_df = meta_df.loc[keep_mask].reset_index(drop=True)
        adverse_df = adverse_df.loc[keep_mask].reset_index(drop=True)
        if x_raw is not None:
            x_raw = x_raw[keep_mask]

        # Reindex final window IDs while preserving candidate_window_idx as an
        # audit link to the original pre-filter candidate sequence.
        final_ids = np.arange(len(y), dtype=int)
        if "super_id" in meta_df.columns:
            meta_df["super_id"] = final_ids
        if "window_idx" in meta_df.columns:
            meta_df["window_idx"] = final_ids
        if "window_idx" in adverse_df.columns:
            adverse_df["window_idx"] = final_ids

        if pma_filter_level == "window":
            if not ((y >= MIN_PMA_WEEKS) & (y <= MAX_PMA_WEEKS)).all():
                raise RuntimeError(f"pid={pid}: PMA window cut left out-of-range y")

        patient_meta["PMA_final_filter_level"] = pma_filter_level
        patient_meta["PMA_final_filter_range_weeks"] = [
            MIN_PMA_WEEKS,
            MAX_PMA_WEEKS,
        ]
        patient_meta["PMA_windows_before_cut"] = n_windows_before_pma
        patient_meta["PMA_windows_removed_by_cut"] = n_windows_removed_pma
        patient_meta["PMA_windows_after_cut"] = int(len(y))
        patient_meta["source_PMA_outside_0_50"] = bool(source_outside)

        stat_row.update(
            status="included",
            reason=None,
            n_windows_before_pma_cut=n_windows_before_pma,
            n_windows_removed_by_pma_cut=n_windows_removed_pma,
            n_windows_after_pma_cut=int(len(y)),
            n_super_after=int(len(y)),
        )
        stats_rows.append(stat_row)

        patient_metadata_by_patient[pid] = patient_meta
        x2_by_patient[pid] = x2
        y_by_patient[pid] = y
        meta_by_patient[pid] = meta_df
        if x_raw is not None:
            x_raw_by_patient[pid] = x_raw
        if INCLUDE_FULL_WINDOW_ADVERSE_METADATA_IN_FINAL_PKL:
            window_adverse_metadata_by_patient[pid] = adverse_df

        json_path = parquet_path.with_name(parquet_path.name + ".cache.json")
        cache_json = read_json_if_available(json_path)
        feature_contract = json_feature_contract(cache_json)
        json_contract_hashes[
            stable_hash(feature_contract)
            if feature_contract is not None
            else "MISSING_JSON"
        ] += 1

    stats_df = pd.DataFrame(stats_rows)
    if len(stats_df):
        stats_df = stats_df.sort_values("pid").reset_index(drop=True)
    removed_df = pd.DataFrame(removed_rows)
    if len(removed_df):
        removed_df = removed_df.sort_values("pid").reset_index(drop=True)

    total_windows = int(sum(len(x) for x in x2_by_patient.values()))
    total_patients = int(len(x2_by_patient))
    total_windows_before_pma = int(
        stats_df.loc[
            stats_df["status"].eq("included"),
            "n_windows_before_pma_cut",
        ].fillna(0).sum()
    ) if len(stats_df) and "n_windows_before_pma_cut" in stats_df.columns else 0

    adverse_counts = Counter()
    for pid in x2_by_patient:
        value = patient_metadata_by_patient.get(pid, {}).get("patient_any_adverse")
        adverse_counts[str(value)] += 1

    final_payload: Dict[str, Any] = {
        "params": {
            **contract,
            "dataset": "USA_external_validation",
            "pipeline_version": PIPELINE_VERSION,
            "id_col": ID_COLUMN,
            "timestamp_col": TIMESTAMP_COLUMN,
            "feature_prefix": "feats__",
            "target_col": SWEDEN_TARGET_COLUMN,
            "static_cols": list(contract["static_feature_cols"]),
            "cohort_name": cohort_name,
            "cohort_GA_min_weeks": MIN_GA_WEEKS,
            "cohort_GA_max_weeks": max_ga_weeks,
            "PMA_filter_min_weeks": MIN_PMA_WEEKS,
            "PMA_filter_max_weeks": MAX_PMA_WEEKS,
            "PMA_filter_level": pma_filter_level,
            "PMA_filter_definition": (
                "patient: remove entire patient if any source PMA is outside range; "
                "window: remove only windows whose target PMA is outside range"
            ),
            "cohort_intubation_filter_applied": False,
            "cohort_intubation_filter_note": (
                "Not applied: no validated USA intubation interval/status field "
                "is referenced by this notebook."
            ),
            "adverse_patients_retained": True,
            "LOS_NEC_patients_retained": True,
            "LOS_NEC_cohort_split_applied": False,
            "LOS_NEC_note": (
                "Old usa_data behavior retained: LOS/NEC are metadata/labels only, "
                "not cohort exclusions."
            ),
            "adverse_group_definition": (
                "adverse if any selected adverse target is positive; control if "
                "no selected adverse target is positive and "
                "patient_exclude_from_control == 0; otherwise "
                "nonadverse_not_strict_control/unknown"
            ),
            "all_static_required": REQUIRE_ALL_STATIC_FEATURES,
            "PMA_target_formula": "feats__ga_w + feats__pna_days / 7",
            "normalization_applied": False,
            "normalization_instruction": (
                "Apply each frozen Sweden checkpoint's saved normalization "
                "during inference; never fit normalization on USA."
            ),
        },
        "feature_cols": list(contract["feature_cols"]),
        "dynamic_feature_cols": list(contract["dynamic_feature_cols"]),
        "static_feature_cols": list(contract["static_feature_cols"]),
        "forbidden_model_features": list(FORBIDDEN_MODEL_FEATURES),
        "X2_by_patient": x2_by_patient,
        "y_by_patient": y_by_patient,
        "meta_by_patient": meta_by_patient,
        "patient_metadata_by_patient": patient_metadata_by_patient,
        "adverse_group_by_patient": {
            pid: meta.get("adverse_group", "unknown")
            for pid, meta in patient_metadata_by_patient.items()
        },
        "patient_ids_by_adverse_group": {
            group: sorted([
                pid for pid, meta in patient_metadata_by_patient.items()
                if meta.get("adverse_group", "unknown") == group
            ])
            for group in [
                "adverse",
                "control",
                "nonadverse_not_strict_control",
                "unknown",
            ]
        },
        "stats_per_patient": stats_df,
        "removed_patients": removed_df,
        "stats_global": {
            "n_input_patient_files": int(len(patient_files)),
            "n_valid_patients": total_patients,
            "n_removed_patients": int(len(patient_files) - total_patients),
            "n_windows_before_pma_cut_in_included_patients": total_windows_before_pma,
            "n_windows_removed_by_pma_cut": int(total_windows_removed_by_pma),
            "n_windows": total_windows,
            "total_super_before": total_windows_before_pma,
            "total_super_after": total_windows,
            "global_retention_ratio": (
                float(total_windows / total_windows_before_pma)
                if total_windows_before_pma > 0
                else np.nan
            ),
            "patients_retained_after_window_cut_despite_source_pma_outlier": int(
                patients_retained_after_window_cut_despite_source_outlier
            ),
            "adverse_flag_counts_valid_patients": dict(adverse_counts),
            "adverse_group_counts_valid_patients": dict(Counter(
                meta.get("adverse_group", "unknown")
                for meta in patient_metadata_by_patient.values()
            )),
            "ga_group_counts_valid_patients": dict(Counter(
                "GA_LE35"
                if float(meta.get("ga_w", np.nan)) <= PRIMARY_MAX_GA_WEEKS
                else "GA_GT35"
                for meta in patient_metadata_by_patient.values()
            )),
            "json_feature_contract_hash_counts": dict(json_contract_hashes),
        },
        "imputation": {
            "method": (
                "per-window per-feature linear interpolation over time + "
                "missingness mask"
            ),
            "output_format": (
                "N,C,T,F with C=2; channel 0=imputed values; "
                "channel 1=1 where original value was missing"
            ),
            "normalization": "none",
        },
        "source": {
            "usa_patient_dir": str(USA_PATIENT_DIR.resolve()),
            "sweden_common_pkl": str(SWEDEN_COMMON_PKL.resolve()),
            "script": (
                str(Path(__file__).resolve())
                if "__file__" in globals()
                else "executed_in_notebook_or_interactive_session"
            ),
        },
    }

    if STORE_RAW_X_BY_PATIENT:
        final_payload["X_by_patient"] = x_raw_by_patient

    final_payload["imputation_stats"] = {
        "total_windows_imputed": total_windows,
        "total_nan_before": int(
            sum(int(np.asarray(x)[:, 1].sum()) for x in x2_by_patient.values())
        ),
        "total_nan_after": 0,
    }
    final_payload["patient_window_filtering_summary"] = {
        "setting_label": cohort_name,
        "superwindow_minutes": contract["superwindow_minutes"],
        "super_stride_minutes": contract["super_stride_minutes"],
        "frames_per_super": contract["frames_per_super"],
        "original_candidate_patient_count_intersection_X_y": int(len(patient_files)),
        "removed_patient_count": int(len(patient_files) - total_patients),
        "final_patient_count": total_patients,
        "original_candidate_X_windows": total_windows_before_pma,
        "original_candidate_y_labels": total_windows_before_pma,
        "final_X_windows": total_windows,
        "final_y_labels": total_windows,
        "PMA_filter_level": pma_filter_level,
    }
    final_payload["x_only_pids"] = []
    final_payload["y_only_pids"] = []

    if INCLUDE_FULL_WINDOW_ADVERSE_METADATA_IN_FINAL_PKL:
        final_payload[
            "window_adverse_metadata_by_patient"
        ] = window_adverse_metadata_by_patient

    return final_payload


# ============================================================================
# 7. AUDIT OUTPUTS
# ============================================================================
def save_audit_outputs(
    final_payload: Dict[str, Any],
    cohort_log_dir: Path,
) -> None:
    cohort_log_dir.mkdir(parents=True, exist_ok=True)

    stats_df = final_payload["stats_per_patient"]
    removed_df = final_payload["removed_patients"]
    patient_meta_df = flatten_patient_metadata_for_csv(
        final_payload["patient_metadata_by_patient"]
    )

    stats_df.to_csv(cohort_log_dir / "USA_patient_window_conversion_summary.csv", index=False)
    removed_df.to_csv(cohort_log_dir / "USA_removed_patients.csv", index=False)
    patient_meta_df.to_csv(cohort_log_dir / "USA_patient_adverse_metadata.csv", index=False)

    feature_df = pd.DataFrame(
        {
            "feature_index": np.arange(len(final_payload["feature_cols"])),
            "feature": final_payload["feature_cols"],
            "role": [
                "dynamic"
                if c in final_payload["dynamic_feature_cols"]
                else "static"
                for c in final_payload["feature_cols"]
            ],
        }
    )
    feature_df.to_csv(cohort_log_dir / "USA_model_feature_contract.csv", index=False)

    atomic_json_dump(
        final_payload["stats_global"],
        cohort_log_dir / "USA_global_conversion_summary.json",
    )
    atomic_json_dump(
        final_payload["params"],
        cohort_log_dir / "USA_conversion_contract.json",
    )

    # Compact cohort table for quick inspection.
    if len(patient_meta_df):
        core_cols = [
            c
            for c in [
                "pid",
                "static_values",
                "ga_w",
                "ga_le_35",
                "adverse_group",
                "patient_any_adverse",
                "patient_strict_control_candidate",
                "patient_event__los",
                "patient_event__abdominal_nec",
                "patient_event__combined_los_nec",
                "patient_first_event_age_hours__los",
                "patient_first_event_age_hours__abdominal_nec",
                "patient_exclude_from_control",
                "patient_label_source",
            ]
            if c in patient_meta_df.columns
        ]
        patient_meta_df[core_cols].to_csv(
            cohort_log_dir / "USA_patient_core_cohort_labels.csv",
            index=False,
        )


# ============================================================================
# 8. REMOVAL DIAGNOSTIC PRINTING
# ============================================================================
def _fmt_number(value: Any, digits: int = 3) -> str:
    try:
        x = float(value)
    except (TypeError, ValueError):
        return "NA"
    return f"{x:.{digits}f}" if np.isfinite(x) else "NA"


def print_removed_patient_reason(
    result: Dict[str, Any],
    patient_index: int,
    n_patients: int,
    cache_hit: bool,
) -> None:
    """Print one compact, informative line for a removed patient."""
    pid = result.get("pid", "UNKNOWN")
    reason = result.get("reason", "unspecified")
    stats = dict(result.get("stats", {}) or {})

    parts = [
        f"[REMOVED {patient_index}/{n_patients}]",
        f"pid={pid}",
        f"reason={reason}",
        f"cache={'hit' if cache_hit else 'rebuilt'}",
    ]

    if stats:
        for key, label in [
            ("n_source_rows", "source_rows"),
            ("n_aligned_unique_rows", "aligned_rows"),
            ("n_duplicate_aligned_rows", "duplicate_aligned_rows"),
            ("n_candidate_windows", "candidate_windows"),
            ("n_rejected_qc", "rejected_QC"),
            ("n_rejected_target", "rejected_target"),
            ("n_windows_passing_qc_before_target", "passed_QC"),
            ("n_kept_windows", "kept_windows"),
        ]:
            if key in stats:
                parts.append(f"{label}={stats[key]}")

        if "window_missing_fraction_median" in stats:
            parts.append(
                "max_feature_missing_fraction[min/median/max]="
                f"{_fmt_number(stats.get('window_missing_fraction_min'))}/"
                f"{_fmt_number(stats.get('window_missing_fraction_median'))}/"
                f"{_fmt_number(stats.get('window_missing_fraction_max'))}"
            )
            parts.append(
                "per_feature_missing_threshold="
                f"{_fmt_number(stats.get('window_missing_fraction_threshold'))}"
            )

        if "source_PMA_min" in stats or "source_PMA_max" in stats:
            parts.append(
                "source_PMA[min/max]="
                f"{_fmt_number(stats.get('source_PMA_min'))}/"
                f"{_fmt_number(stats.get('source_PMA_max'))}"
            )

    print(" | ".join(parts), flush=True)


# ============================================================================
# 9. MAIN
# ============================================================================
def main() -> None:
    require_parquet_engine()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PATIENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    outputs = {
        "ALL_GA_PMA50_PATIENT_CUT": FINAL_OUTPUT_ALL_GA_PMA50_PATIENT_CUT_PKL,
        "ALL_GA_PMA50_WINDOW_CUT": FINAL_OUTPUT_ALL_GA_PMA50_WINDOW_CUT_PKL,
        "GA_LE35_PMA50_PATIENT_CUT": FINAL_OUTPUT_GA_LE35_PMA50_PATIENT_CUT_PKL,
        "GA_LE35_PMA50_WINDOW_CUT": FINAL_OUTPUT_GA_LE35_PMA50_WINDOW_CUT_PKL,
    }
    if MAX_PATIENTS is not None:
        outputs = {
            label: path.with_name(
                f"{path.stem}_SMOKE_{int(MAX_PATIENTS)}{path.suffix}"
            )
            for label, path in outputs.items()
        }

    existing = [path for path in outputs.values() if path.exists()]
    if existing and not OVERWRITE_FINAL_PKL:
        raise FileExistsError(
            "Final output already exists:\n"
            + "\n".join(map(str, existing))
            + "\nSet OVERWRITE_FINAL_PKL=True only if replacement is intentional."
        )

    contract = load_sweden_contract(SWEDEN_COMMON_PKL)
    print("=" * 100)
    print("SWEDEN MODEL INPUT CONTRACT")
    print("=" * 100)
    for key in [
        "frame_len_minutes",
        "frame_hop_minutes",
        "superwindow_minutes",
        "super_stride_minutes",
        "frames_per_super",
        "align_method",
    ]:
        print(f"{key}: {contract[key]}")
    print(
        "USA QC rule: every dynamic feature must have observed-frame ratio "
        f">= {FALLBACK_VALID_RATIO:.0%} within each 23-frame superwindow"
    )
    print("Feature order:")
    for i, col in enumerate(contract["feature_cols"]):
        print(f"  {i:02d} {col}")

    patient_files = sorted(USA_PATIENT_DIR.glob("*.parquet"))
    if not patient_files:
        raise FileNotFoundError(f"No *.parquet files found in {USA_PATIENT_DIR}")
    if MAX_PATIENTS is not None:
        patient_files = patient_files[: int(MAX_PATIENTS)]

    print("\n" + "=" * 100)
    print("USA PATIENT CONVERSION")
    print("=" * 100)
    print("Patient directory:", USA_PATIENT_DIR)
    print("Patients selected:", len(patient_files))
    print("Patient cache directory:", PATIENT_CACHE_DIR)
    print("Adverse / LOS / NEC patients retained in every output:", True)
    print("GA processed range:", MIN_GA_WEEKS, "to no upper limit")
    print("GA threshold for restricted outputs:", PRIMARY_MAX_GA_WEEKS)
    print("PMA cut range:", MIN_PMA_WEEKS, "to", MAX_PMA_WEEKS)
    print("PMA cut variants: entire patient and individual window")

    start_clock = time.time()
    status_counts = Counter()
    reason_counts = Counter()
    cache_hits = 0
    n_removed_details_printed = 0

    for i, parquet_path in enumerate(patient_files, start=1):
        result, cache_hit = convert_one_patient_file(parquet_path, contract)
        status = result.get("status", "unknown")
        reason = result.get("reason", "")
        status_counts[status] += 1
        if reason:
            reason_counts[reason] += 1
        cache_hits += int(cache_hit)

        if status == "removed" and PRINT_REMOVED_PATIENT_DETAILS:
            within_limit = (
                PRINT_REMOVED_PATIENT_LIMIT is None
                or n_removed_details_printed < int(PRINT_REMOVED_PATIENT_LIMIT)
            )
            if within_limit:
                print_removed_patient_reason(
                    result=result,
                    patient_index=i,
                    n_patients=len(patient_files),
                    cache_hit=cache_hit,
                )
                n_removed_details_printed += 1

        if i == 1 or i % PRINT_EVERY_N_PATIENTS == 0 or i == len(patient_files):
            elapsed_min = (time.time() - start_clock) / 60.0
            print(
                f"[{i}/{len(patient_files)}] "
                f"status={dict(status_counts)} | "
                f"cache_hits={cache_hits} | "
                f"elapsed={elapsed_min:.1f} min"
            )

    print("\nConversion status counts:", dict(status_counts))
    print("Removal reason counts before final GA/PMA cohort cuts:", dict(reason_counts))
    if PRINT_REMOVED_PATIENT_DETAILS:
        print("Detailed removed-patient lines printed:", n_removed_details_printed)

    print("\n" + "=" * 100)
    print("CONSOLIDATING FOUR FINAL PKLS")
    print("=" * 100)

    payloads = {
        "ALL_GA_PMA50_PATIENT_CUT": consolidate_results(
            patient_files,
            contract,
            cohort_name="ALL_GA_PMA50_PATIENT_CUT",
            max_ga_weeks=None,
            pma_filter_level="patient",
        ),
        "ALL_GA_PMA50_WINDOW_CUT": consolidate_results(
            patient_files,
            contract,
            cohort_name="ALL_GA_PMA50_WINDOW_CUT",
            max_ga_weeks=None,
            pma_filter_level="window",
        ),
        "GA_LE35_PMA50_PATIENT_CUT": consolidate_results(
            patient_files,
            contract,
            cohort_name="GA_LE35_PMA50_PATIENT_CUT",
            max_ga_weeks=PRIMARY_MAX_GA_WEEKS,
            pma_filter_level="patient",
        ),
        "GA_LE35_PMA50_WINDOW_CUT": consolidate_results(
            patient_files,
            contract,
            cohort_name="GA_LE35_PMA50_WINDOW_CUT",
            max_ga_weeks=PRIMARY_MAX_GA_WEEKS,
            pma_filter_level="window",
        ),
    }

    for label, payload in payloads.items():
        save_audit_outputs(payload, LOG_DIR / label)
        print(
            f"{label}: patients={payload['stats_global']['n_valid_patients']}, "
            f"windows={payload['stats_global']['n_windows']}, "
            f"PMA_windows_removed={payload['stats_global']['n_windows_removed_by_pma_cut']}, "
            f"adverse_groups={payload['stats_global']['adverse_group_counts_valid_patients']}"
        )
        for pid, x in payload["X2_by_patient"].items():
            if x.shape[1:] != (2, 23, 17):
                raise RuntimeError(f"{label} pid={pid}: unexpected shape {x.shape}")
            if len(x) != len(payload["y_by_patient"][pid]):
                raise RuntimeError(f"{label} pid={pid}: final X/y mismatch")

    all_patient = payloads["ALL_GA_PMA50_PATIENT_CUT"]
    all_window = payloads["ALL_GA_PMA50_WINDOW_CUT"]
    le35_patient = payloads["GA_LE35_PMA50_PATIENT_CUT"]
    le35_window = payloads["GA_LE35_PMA50_WINDOW_CUT"]

    ids_all_patient = set(all_patient["X2_by_patient"])
    ids_all_window = set(all_window["X2_by_patient"])
    ids_le35_patient = set(le35_patient["X2_by_patient"])
    ids_le35_window = set(le35_window["X2_by_patient"])

    if not ids_all_patient.issubset(ids_all_window):
        raise RuntimeError("ALL_GA patient-cut patients must be a subset of window-cut patients")
    if not ids_le35_patient.issubset(ids_le35_window):
        raise RuntimeError("GA_LE35 patient-cut patients must be a subset of window-cut patients")
    if not ids_le35_patient.issubset(ids_all_patient):
        raise RuntimeError("GA_LE35 patient-cut cohort is not a subset of ALL_GA patient-cut")
    if not ids_le35_window.issubset(ids_all_window):
        raise RuntimeError("GA_LE35 window-cut cohort is not a subset of ALL_GA window-cut")

    for label in ["GA_LE35_PMA50_PATIENT_CUT", "GA_LE35_PMA50_WINDOW_CUT"]:
        for pid, meta in payloads[label]["patient_metadata_by_patient"].items():
            if float(meta["ga_w"]) > PRIMARY_MAX_GA_WEEKS:
                raise RuntimeError(f"{label} contains GA>35 patient: {pid}")

    for label in ["ALL_GA_PMA50_PATIENT_CUT", "GA_LE35_PMA50_PATIENT_CUT"]:
        for pid, meta in payloads[label]["patient_metadata_by_patient"].items():
            if (
                float(meta["source_PMA_min"]) < MIN_PMA_WEEKS
                or float(meta["source_PMA_max"]) > MAX_PMA_WEEKS
            ):
                raise RuntimeError(f"{label} contains PMA-outlier patient: {pid}")

    for label in ["ALL_GA_PMA50_WINDOW_CUT", "GA_LE35_PMA50_WINDOW_CUT"]:
        for pid, y in payloads[label]["y_by_patient"].items():
            if not ((y >= MIN_PMA_WEEKS) & (y <= MAX_PMA_WEEKS)).all():
                raise RuntimeError(f"{label} contains PMA-outlier window: {pid}")

    for label, payload in payloads.items():
        atomic_pickle_dump(payload, outputs[label])

    print("\n" + "=" * 100)
    print("DONE")
    print("=" * 100)
    for label, path in outputs.items():
        print(f"{label}:\n  {path}")
    print("Audit logs:")
    print(LOG_DIR)
    print("\nLOS/NEC behavior: retained in all four outputs as metadata only.")
    print("Do not normalize USA here. During inference, apply each frozen")
    print("Sweden checkpoint's saved mean/std values.")


if __name__ == "__main__":
    main()